## <u>**VSCode: Polyglot Notebooks**</u>



**Polyglot Notebooks** is an extension for Visual Studio Code that allows you to create and work with notebooks that support multiple programming languages within a single document. This is particularly useful for data scientists, researchers, and developers who often need to combine code from different languages in their workflows.
It supports languages such as **PowerShell**, **Python**, **R**, **JavaScript**, and more, enabling seamless integration and execution of code cells in different languages.

1. To use a language in a cell, you typically start the cell with a magic command that specifies the language.
2. For example, to use PowerShell in a cell, you would start the cell with `#!pwsh`.
3. So an example of a PowerShell cell in a Polyglot Notebook would look like this:

```pwsh
#!pwsh
Get-ChildItem -Path "C:\Your\Directory\Path" -Recurse | Where-Object { $_.Extension -eq ".txt" } | Select-Object FullName, Length, LastWriteTime
```

----

<br>

## PowerShell Notebooks - Executable Documents

### Loops

```pwsh
#!pwsh
1..5 | % {
    $_ * 2
}

2
4
6
8
10
```

### Continued:

1. List all .txt files in specified directory & its subdirectories, displaying their full names, sizes, & last write times.

In [ ]:
#!pwsh
Get-ChildItem -Path "C:\Users\josep\Documents" | Where-Object { $_.LastWriteTime -lt (Get-Date).AddDays(-30) }

2. Get folder utilization for given directory.

In [ ]:
#!pwsh

Dir -path C:\Users\josep\Documents\Warp -file -recurse -force |
  Measure-Object length -sum -max -average |
  Select-Object @{name='Total Files';Expression={$_.count}},
    @{name='Largest File(MB)';Expression={'{0:F2}' -f ($_.maximum/1MB)}},
    @{name='Average Size(MB)';Expression={'{0:F2}' -f ($_.average/1MB)}},
    @{name='Total Size(MB)';Expression={'{0:F2}' -f ($_.sum/1MB)}}

3. Added a custom key binding to display my custom keybindings using `Format-SpectreTable` from the **pwshSpectreConsole** module

In [13]:
#!pwsh
Set-PSReadLineKeyHandler -Chord "Alt+k" -BriefDescription "my handlers" -Description "Show my custom PSReadline key handlers" -ScriptBlock {
    Get-PSReadLineKeyHandler -bound |
    Where group -eq 'Custom' |
    Sort Key |
    Format-SpectreTable -Title "[italic gold1]My PSReadLine Key Handlers[/]" -Color 'SpringGreen3_1' |
    Out-Host
}

In [14]:
#!pwsh
$wtProfilePath = Join-Path -Path $env:LocalAppData -ChildPath 'Packages\Microsoft.WindowsTerminal_8wekyb3d8bbwe\LocalState\settings.json'
$wtProfile = Get-Content $wtProfilePath | ConvertFrom-Json
$wtProfile.keybindings

key,value
id,Terminal.CopyToClipboard
keys,ctrl+c


key,value
id,Terminal.PasteFromClipboard
keys,ctrl+v


key,value
id,Terminal.DuplicatePaneAuto
keys,alt+shift+d


---

---

## Extending .NET Interactive with Custom Extensions

The ClockExtension sample illustrates this approach. There's also walkthrough in the form of a notebook that shows you how to build and install it.

[This PC - ClockExtension.ipynb](.\ClockExtension.ipynb)\
[Github - extending-dotnet-interactive](https://github.com/dotnet/interactive/blob/main/docs/extending-dotnet-interactive.md)

In [6]:
// Copyright (c) .NET Foundation and contributors. All rights reserved.
// Licensed under the MIT license. See LICENSE file in the project root for full license information.

using System;
using System.Threading.Tasks;
using Microsoft.DotNet.Interactive;
using Microsoft.DotNet.Interactive.Directives;
using Microsoft.DotNet.Interactive.Formatting;
using static Microsoft.DotNet.Interactive.Formatting.PocketViewTags;

namespace ClockExtension;

public static class ClockKernelExtension
{
    public static void Load(Kernel kernel)
    {
        // Register formatters that will change the default output when formatting DateTime and DateTimeOffset instances as text/html.
        Formatter.Register<DateTime>((date, writer) => writer.Write(date.DrawSvgClock()), "text/html");

        Formatter.Register<DateTimeOffset>((date, writer) => writer.Write(date.DrawSvgClock()), "text/html");

        // Next, define a magic command that will render a clock.
        var hourParameter = new KernelDirectiveParameter("--hour", "The position of the hour hand");
        var minuteParameter = new KernelDirectiveParameter("--minute", "The position of the minute hand");
        var secondParameter = new KernelDirectiveParameter("--second", "The position of the second hand");

        var clockDirective = new KernelActionDirective("#!clock")
        {
            Description = "Displays a clock showing the current or specified time.",
            Parameters =
            [
                hourParameter,
                minuteParameter,
                secondParameter
            ]
        };

        kernel.AddDirective<DisplayClock>(clockDirective, (displayClock, context) =>
        {
            context.Display(SvgClock.DrawSvgClock(displayClock.Hour, displayClock.Minute, displayClock.Second));
            return Task.CompletedTask;
        });

        // Finally, display some information to the user so they can see how to use the extension.
        PocketView view = div(
            code(nameof(ClockExtension)),
            " is loaded. It adds visualizations for ",
            code(typeof(DateTime)),
            " and ",
            code(typeof(DateTimeOffset)),
            ". Try it by running: ",
            code("DateTime.Now")
        );

        KernelInvocationContext.Current?.Display(view);
    }
}

Error: (11,1): error CS7021: Cannot declare namespace in script code